# 03 — Generate baseline predictions

**Purpose:** use the fine-tuned checkpoint from Notebook 02 to generate one fixed baseline summary per abstract using greedy decoding.

**Your responsibility:** provide reproducible baseline predictions. This notebook does not generate five candidates, calculate AlignScore, perform reranking, or calculate SummaC; those belong to the later reranking/evaluation stage.

**Outputs:** `outputs/val_baseline.csv` and, after the experiment settings are frozen, `outputs/test_baseline.csv`.

The validation predictions can be used while developing the experiment. The final test predictions should be generated only after the model and decoding settings are fixed.


In [1]:
from pathlib import Path
import json
import pandas as pd
import torch
import yaml
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = yaml.safe_load((ROOT / "configs" / "config.yaml").read_text())
D, B = CFG["data"], CFG["baseline"]
MODEL_DIR = ROOT / "outputs" / "model"
OUT = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

if not (MODEL_DIR / "config.json").exists():
    raise FileNotFoundError("Fine-tuned checkpoint not found. Run Notebook 02 first.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


## 1. Load the fine-tuned model and tokenizer


In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device)
model.eval()

def read_split(name):
    path = ROOT / "data" / "processed" / f"{name}.jsonl"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run Notebook 01 first.")
    return pd.read_json(path, orient="records", lines=True)

val_df = read_split("val")
print("Validation papers:", len(val_df))


Validation papers: 617


## 2. Set the fixed baseline decoding configuration

The baseline is greedy decoding (`num_beams=1`, `do_sample=False`) and returns exactly one summary per abstract. These settings are separate from training hyperparameters.


In [3]:
decode = dict(B["decoding"])
assert decode["num_beams"] == 1 and decode["do_sample"] is False
decode["num_return_sequences"] = 1
print("Baseline decoding:", decode)

# Use a smaller inference batch on small GPUs; this changes speed/memory, not decoding.
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    batch_size = min(int(B["batch_size"]), 4 if vram_gb < 8 else int(B["batch_size"]))
else:
    batch_size = 1
print("Inference batch size:", batch_size)


Baseline decoding: {'num_beams': 1, 'do_sample': False, 'max_new_tokens': 64, 'no_repeat_ngram_size': 3, 'num_return_sequences': 1}
Inference batch size: 4


## 3. Generate baseline summaries

Each processed paper gets one generated TLDR. The paper ID and all human references are retained in the output CSV for traceability and later multi-reference evaluation.


In [4]:
import csv

def generate_predictions(frame, split_name):
    records = []
    abstracts = frame["abstract"].tolist()

    for start in tqdm(range(0, len(abstracts), batch_size), desc=f"{split_name} baseline"):
        batch_abstracts = abstracts[start:start + batch_size]
        encoded = tokenizer(
            batch_abstracts,
            max_length=D["max_source_len"],
            truncation=True,
            padding=True,
            return_tensors="pt"
        ).to(device)

        with torch.inference_mode():
            generated = model.generate(
                input_ids=encoded["input_ids"],
                attention_mask=encoded["attention_mask"],
                **decode
            )

        summaries = tokenizer.batch_decode(generated, skip_special_tokens=True)
        for offset, summary in enumerate(summaries):
            row = frame.iloc[start + offset]
            records.append({
                "id": row["id"],
                "abstract": row["abstract"].replace("\n", " ").replace("\r", ""),
                "generated_tldr": summary.strip().replace("\n", " "),
                "human_tldrs": json.dumps(row["human_tldrs"], ensure_ascii=False)
            })

    return pd.DataFrame(records)

val_predictions = generate_predictions(val_df, "validation")
val_path = OUT / "val_baseline.csv"
val_predictions.to_csv(val_path, index=False, quoting=csv.QUOTE_MINIMAL)
print("Saved:", val_path)
display(val_predictions[["id", "generated_tldr"]].head(10))
print("Generated validation summaries:", len(val_predictions))


## 4. Inspect basic output properties

These are descriptive checks, not substitutes for AlignScore or SummaC.


In [5]:
val_predictions["generated_words"] = val_predictions["generated_tldr"].str.split().str.len()
val_predictions["generated_sentences_approx"] = val_predictions["generated_tldr"].str.count(r"[.!?]+")
display(val_predictions[["generated_words", "generated_sentences_approx"]].describe().round(2))
print("Empty generations:", int(val_predictions["generated_tldr"].str.strip().eq("").sum()))
print("Saved validation baseline:", val_path)


## 5. Generate final test predictions only after freezing the experiment

Do not run this cell until your team has fixed the model checkpoint and all baseline decoding settings using the validation stage. The processed test set may contain fewer than the official 618 examples because Notebook 01 removes cross-split duplicate abstracts; use the actual processed count.


In [ ]:
RUN_FINAL_TEST = False  # Change to True only after model/settings are frozen.

if RUN_FINAL_TEST:
    test_df = read_split("test")
    test_predictions = generate_predictions(test_df, "test")
    test_path = OUT / "test_baseline.csv"
    test_predictions.to_csv(test_path, index=False, quoting=csv.QUOTE_MINIMAL)
    print("Saved:", test_path)
    print("Final processed test predictions:", len(test_predictions))
else:
    print("Test generation skipped. Set RUN_FINAL_TEST = True only for the final locked test run.")


## Deliverables to share with your teammates

- `outputs/model/` — fine-tuned BART model and tokenizer.
- `outputs/val_baseline.csv` — one greedy baseline prediction per validation abstract.
- `outputs/test_baseline.csv` — final test baseline, created only after settings are frozen.

The `human_tldrs` column is JSON-encoded so all references are preserved. Teammates can parse it with `json.loads()`.

**Not included here:** five-candidate generation, candidate-diversity analysis, AlignScore scoring/selection, and SummaC evaluation. These are separate downstream tasks.
